In [ ]:
import json
import glob

grouped_folders = {}

for file in glob.glob("*folders.json"):
    prefix = file.replace("folders.json", "")
    with open(file, "r") as f:
        grouped_folders[prefix] = json.load(f)


In [ ]:
# ============================================================
# --- ConceptNet Local Database Setup ---
# ============================================================

import sqlite3
from pathlib import Path

DB_FILE = Path.home() / "conceptnet_data" / "conceptnet.db"
TOP_K_EDGES = 20

RELATION_MAP = {
    "IsA": "/r/IsA",
    "PartOf": "/r/PartOf",
    "HasA": "/r/HasA",
    "UsedFor": "/r/UsedFor",
    "CapableOf": "/r/CapableOf",
    "AtLocation": "/r/AtLocation",
    "LocatedNear": "/r/LocatedNear",
    "MadeOf": "/r/MadeOf",
    "SimilarTo": "/r/SimilarTo",
    "RelatedTo": "/r/RelatedTo",
    "HasProperty": "/r/HasProperty",
    "CreatedBy": "/r/CreatedBy",
    "HasPrerequisite": "/r/HasPrerequisite",
    "HasSubevent": "/r/HasSubevent",
    "HasFirstSubevent": "/r/HasFirstSubevent",
    "HasLastSubevent": "/r/HasLastSubevent",
    "MotivatedByGoal": "/r/MotivatedByGoal"
}


def concept_uri(term: str):
    return f"/c/en/{term.strip().replace(' ', '_')}"


def extract_label(uri: str):
    parts = uri.split("/")
    if len(parts) >= 4:
        return parts[-1].replace("_", " ")
    return uri


def query_edges(conn, uri, relation, limit=TOP_K_EDGES):
    cursor = conn.cursor()

    # First try forward edges
    cursor.execute(
        """
        SELECT relation, start, end, weight
        FROM edges
        WHERE start = ? AND relation = ?
        ORDER BY weight DESC
        LIMIT ?
        """,
        (uri, relation, limit),
    )

    rows = cursor.fetchall()

    if rows:
        return rows

    # Fallback: any direction
    cursor.execute(
        """
        SELECT relation, start, end, weight
        FROM edges
        WHERE (start = ? OR end = ?) AND relation = ?
        ORDER BY weight DESC
        LIMIT ?
        """,
        (uri, uri, relation, limit),
    )

    return cursor.fetchall()

def ConceptSearch(concept):
    """
    Search ConceptNet for any relations involving the concept.
    Returns top edges across relations.
    """

    uri = concept_uri(concept)

    if not DB_FILE.exists():
        return "ConceptNet database not found."

    conn = sqlite3.connect(DB_FILE)

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT relation, start, end, weight
            FROM edges
            WHERE start = ? OR end = ?
            ORDER BY weight DESC
            LIMIT ?
            """,
            (uri, uri, TOP_K_EDGES),
        )

        rows = cursor.fetchall()

        if not rows:
            return f"No ConceptNet knowledge found for '{concept}'."

        outputs = []

        for rel, start, end, weight in rows:

            start_label = extract_label(start)
            end_label = extract_label(end)

            # convert /r/IsA → IsA
            rel_label = rel.split("/")[-1]

            if start == uri:
                direction = "→"
            elif end == uri:
                direction = "←"
            else:
                direction = "↔"

            weight_str = f"{weight:.2f}" if weight is not None else "N/A"

            outputs.append(
                f"{start_label} --{rel_label}--> {end_label} (w={weight_str})"
            )

        return "\n".join(outputs)

    finally:
        conn.close()

def ConceptRelate(concept, relation):
    """
    Local ConceptNet query replacing the API version.
    Returns formatted edges for the LLM.
    """

    if relation not in RELATION_MAP:
        return f"No relation mapping for {relation}"

    rel_uri = RELATION_MAP[relation]
    uri = concept_uri(concept)

    if not DB_FILE.exists():
        return "ConceptNet database not found."

    conn = sqlite3.connect(DB_FILE)

    try:
        rows = query_edges(conn, uri, rel_uri, TOP_K_EDGES)

        if not rows:
            return f"No edges found for ({concept}, {relation})"

        outputs = []

        for rel, start, end, weight in rows:

            start_label = extract_label(start)
            end_label = extract_label(end)

            if start == uri:
                direction = "→"
            elif end == uri:
                direction = "←"
            else:
                direction = "↔"

            # SAFE weight formatting
            weight_str = f"{weight:.2f}" if weight is not None else "N/A"

            outputs.append(
                f"{start_label} --{relation}--> {end_label} (w={weight_str})"
            )

        return "\n".join(outputs)

    finally:
        conn.close()

def ConceptRelateReverse(concept, relation):
    """
    Query ConceptNet for reverse relations.
    Example:
        ConceptRelateReverse("clean", "UsedFor")

    Returns:
        ? --UsedFor--> clean
    """

    if relation not in RELATION_MAP:
        return f"No relation mapping for {relation}"

    rel_uri = RELATION_MAP[relation]
    uri = concept_uri(concept)

    if not DB_FILE.exists():
        return "ConceptNet database not found."

    conn = sqlite3.connect(DB_FILE)

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT relation, start, end, weight
            FROM edges
            WHERE end = ? AND relation = ?
            ORDER BY weight DESC
            LIMIT ?
            """,
            (uri, rel_uri, TOP_K_EDGES),
        )

        rows = cursor.fetchall()

        if not rows:
            return f"No reverse edges found for ({relation} → {concept})"

        outputs = []

        for rel, start, end, weight in rows:

            start_label = extract_label(start)
            end_label = extract_label(end)

            weight_str = f"{weight:.2f}" if weight is not None else "N/A"

            outputs.append(
                f"{start_label} --{relation}--> {end_label} (w={weight_str})"
            )

        return "\n".join(outputs)

    finally:
        conn.close()

In [ ]:
ConceptRelate("tomato", "AtLocation")

In [ ]:
ConceptSearch("countertop")

In [ ]:
import os
import json
import glob
import random
import re
import textworld
import requests
from os.path import join as pjoin
from wordsegment import load, segment

from alfworld.info import ALFWORLD_DATA
from alfworld.agents.utils.misc import add_task_to_grammar
from alfworld.agents.environment.alfred_tw_env import AlfredExpert, AlfredDemangler, AlfredExpertType
from groq import Groq

# ============================================================
# --- 0️⃣ LLM and ENVIRONMENT SETUP ---
# ============================================================
try:

    client = Groq(api_key=os.getenv("API KEY NAME HERE", "API KEY PLACEHOLDER"))


#
    LLM_MODEL = "llama-3.3-70b-versatile"
    print(f"✅ Groq client initialized with model: {LLM_MODEL}")
except Exception:
    print("❌ Failed to initialize Groq client. Please set the GROQ_API_KEY environment variable.")
    exit()

load()
print("✅ Word segmentation library initialized.")

ALL_CONCEPTNET_RELATIONS = [
    "IsA",     "PartOf",     "HasA",     "UsedFor",     "CapableOf",     "AtLocation",     "LocatedNear",     "MadeOf",
    "SimilarTo",     "RelatedTo",     "HasProperty",     "CreatedBy",     "HasPrerequisite",     "HasSubevent",     "HasFirstSubevent",     "HasLastSubevent",
    "MotivatedByGoal" ]
# ============================================================
# --- CORE GAME AND TOOL FUNCTIONS (UNCHANGED) ---
# ============================================================
def initialize_alfworld_env(task_type, task_id):
    class Args:
        problem = None
        domain = pjoin(ALFWORLD_DATA, "logic", "alfred.pddl")
        grammar = pjoin(ALFWORLD_DATA, "logic", "alfred.twl2")
    args = Args()
    args.problem = f"/home/sheema/.cache/alfworld/json_2.1.1/valid_seen/{task_type}/{task_id}"

    print(f"🎮 Starting ALFWorld game: '{os.path.basename(args.problem)}'")
    GAME_LOGIC = {"pddl_domain": open(args.domain).read(), "grammar": open(args.grammar).read()}
    pddl_file = pjoin(args.problem, "initial_state.pddl")
    json_file = pjoin(args.problem, "traj_data.json")
    with open(json_file, "r") as f:
        traj_data = json.load(f)
    GAME_LOGIC["grammar"] = add_task_to_grammar(GAME_LOGIC["grammar"], traj_data)
    gamedata = dict(**GAME_LOGIC, pddl_problem=open(pddl_file).read())
    gamefile = pjoin(os.path.dirname(pddl_file), "game.tw-pddl")
    json.dump(gamedata, open(gamefile, "w"))
    expert = AlfredExpert(expert_type=AlfredExpertType.HANDCODED)
    request_infos = textworld.EnvInfos(won=True, admissible_commands=True, extras=["expert_plan"])
    env_id = textworld.gym.register_game(
        gamefile, request_infos, max_episode_steps=100, wrappers=[AlfredDemangler(), expert]
    )
    env = textworld.gym.make(env_id)
    obs, infos = env.reset()
    print("\n" + obs.strip())
    print("-" * 60)
    return env, obs

def play_one_step(env, command):
    obs, score, done, infos = env.step(command)
    return obs, done, infos

def segment_concept(concept: str) -> str:
    if " " not in concept.strip():
        segmented_parts = segment(concept)
        if len(segmented_parts) > 1:
            new_concept = " ".join(segmented_parts)
            print(f"   (Segmented concept: '{concept}' -> '{new_concept}')")
            return new_concept
    return concept


# ============================================================
# --- NEW "ReAct" AGENT BRAIN ---
# ============================================================
def get_next_action(history, examples):
    ALL_CONCEPTNET_RELATIONS = [
    "IsA",
    "PartOf",
    "HasA",
    "UsedFor",
    "CapableOf",
    "AtLocation",
    "LocatedNear",
    "MadeOf",
    "SimilarTo",
    "RelatedTo",
    "HasProperty",
    "CreatedBy",
    "HasPrerequisite",
    "HasSubevent",
    "HasFirstSubevent",
    "HasLastSubevent",
    "MotivatedByGoal"
]

    ALFWorld_Action_Space = {
    "look": "Observe the current surroundings",
    "inventory": "Check the agent’s current state and held object",
    "go to <receptacle>": "Move to a specific location",
    "open <receptacle>": "Open an openable receptacle",
    "close <receptacle>": "Close an openable receptacle",
    "take <object> from <receptacle>": "Pick up an object from a receptacle",
    "move <object> to <receptacle>": "Place a held object into a receptacle",
    "examine <something>": "Inspect the state of an object or receptacle",
    "use <object>": "Use an object to perform an action",
    "heat <object> with <receptacle>": "Heat an object using a heating receptacle",
    "clean <object> with <receptacle>": "Clean an object using a cleaning receptacle",
    "cool <object> with <receptacle>": "Cool an object using a cooling receptacle",
    "slice <object> with <object>": "Cut an object using a tool"
}

    """
    This is the core of the ReAct agent. It takes the game history and
    decides on the SINGLE next action to take.
    """
    system_prompt = """
You are an expert planner agent playing the text-based game ALFWorld. Your goal is to complete the household task by reacting to observations and manipulating the game state step by step.

You must determine whether the game has just started or whether the agent is already mid-execution, and decide the next move accordingly. In all scenarios, you MUST learn from the provided {examples} when choosing your next action.

You MUST NOT use internal or external world knowledge to invent objects, receptacles, or actions. All reasoning and actions must be strictly grounded in the MOST RECENT observation. If a required object or receptacle is not present in the observation, you must continue searching and MUST NOT hallucinate.

Remember, the agent can only hold a single object at a time. So if you are moving two objects, you MUST move one by one.

When you need information to achieve a subgoal, you MUST retrieve it via ConceptNet using the allowed {ALFWorld_Action_Space}. You must then use the retrieved information, following the patterns in {examples}, to guide your decision-making.

When you call ConceptNet, depending on the current task, you should also decide which relations are relevant to retrieve. You can see ALL_CONCEPTNET_RELATIONS to decide.

Every action you perform MUST:
- Manipulate the game state
- Be a valid action from {ALFWorld_Action_Space}
- Preserve correct object and receptacle identifiers

Following the {examples}, you MAY perform multiple actions in a single step if appropriate.

After each action, you MUST reason about the resulting state:
- If an action succeeds, proceed toward the next goal or subgoal
- If an action fails, you MUST NOT repeat it and should attempt an alternative valid action
- If you are already holding an object, you can not pick another one. You should first move your current object somewhere in order to take the other one.

When the observation explicitly states that the main goal has been completed, you MUST stop immediately and perform no further actions.

Your response MUST follow this format and NOTHING else:

LLM Think: [Your reasoning grounded strictly in the latest observation and any retrieved ConceptNet information.]
Your command: [Your single command, which may be a game action or a ConceptNet query.]

"""

    example_texts = [f"--- EXAMPLE {i} START ---\n{value}\n--- EXAMPLE {i} END ---" for i, (key, value) in enumerate(examples.items(), 1)]
    all_examples_string = "\n\n".join(example_texts)

    user_prompt = f"""
Here are examples of how to solve tasks.
{all_examples_string}

Now, continue the following game. Provide only your next thought and command.

--- CURRENT GAME HISTORY ---
{history}
--- END OF HISTORY ---

Your next step:
"""

    completion = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=0.1,
    )
    return completion.choices[0].message.content.strip()

def fetch_concept_edges_from_relations(concept, relations):
    outputs = []
    for rel in relations:
        outputs.append(ConceptRelate(concept, rel))
    return "\n".join(outputs)


In [ ]:
import os

BASE_PATH = "/home/sheema/.cache/alfworld/json_2.1.1/valid_seen"

def get_trial_ids(task_type):
    task_path = os.path.join(BASE_PATH, task_type)

    return [
        d for d in os.listdir(task_path)
        if d.startswith("trial_") and os.path.isdir(os.path.join(task_path, d))
    ]

import json
import os

PROGRESS_FILE = "alfworld_progress.json"

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f).get("task_index", 0)
    return 0


In [ ]:
import os
import json
import glob
import re
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

Approach = 'Main'

BASE_PATH = "/home/sheema/.cache/alfworld/json_2.1.1/valid_seen"
PREFIX = "pick_heat_then_place_"
PROGRESS_FILE = f"progress_{PREFIX}_{Approach}.json"
MAX_STEPS = 50
CSV_LOG_FILE = f"{Approach}_success_ratio_{PREFIX}.csv"

# ============================================================
# PROGRESS UTILS
# ============================================================

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f).get("task_index", 0)
    return 0

def save_progress(task_index):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"task_index": task_index}, f)

# ============================================================
# TASK QUEUE BUILDER
# ============================================================

def build_task_queue(grouped_folders, prefix):
    queue = []
    for task_type in grouped_folders[prefix]:
        task_path = os.path.join(BASE_PATH, task_type)
        if not os.path.isdir(task_path):
            continue

        trial_ids = sorted([
            d for d in os.listdir(task_path)
            if d.startswith("trial_") and os.path.isdir(os.path.join(task_path, d))
        ])

        for task_id in trial_ids:
            queue.append((task_type, task_id))

    return queue

# ============================================================
# SUCCESS RATIO LOGGING (CSV)
# ============================================================

def log_task_result(task_id, task_type, success, steps):
    # Load existing CSV or create empty DF
    if os.path.exists(CSV_LOG_FILE):
        df = pd.read_csv(CSV_LOG_FILE)
    else:
        df = pd.DataFrame(columns=["task_id", "task_type", "success", "steps"])

    # Check if this task already exists
    mask = (df["task_id"] == task_id) & (df["task_type"] == task_type)

    if mask.any():
        # 🔁 Overwrite existing row
        df.loc[mask, "success"] = success
        df.loc[mask, "steps"] = steps
        print(f"♻️ Updated existing entry for {task_type}/{task_id}")
    else:
        # ➕ Add new row
        df = pd.concat([df, pd.DataFrame([{
            "task_id": task_id,
            "task_type": task_type,
            "success": success,
            "steps": steps
        }])], ignore_index=True)
        print(f"➕ Added new entry for {task_type}/{task_id}")

    # Save back
    df.to_csv(CSV_LOG_FILE, index=False)
    print(f"💾 Saved → Success: {success}, Steps: {steps}")
# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # ----------------------------
    # Load few-shot examples
    # ----------------------------
    task_prefix = "demo-heat"
    file_list = glob.glob(f"{task_prefix}*.txt")

    if not file_list:
        print(f"❌ No few-shot files found with prefix '{task_prefix}'")
        exit()

    few_shot_examples = {}
    for filepath in file_list:
        key = os.path.basename(filepath).replace(".txt", "")
        with open(filepath, "r") as f:
            few_shot_examples[key] = f.read()

    print("✅ Loaded examples:", list(few_shot_examples.keys()))

    # ----------------------------
    # Build task queue
    # ----------------------------
    task_queue = build_task_queue(grouped_folders, PREFIX)
    print(f"📦 Total tasks discovered: {len(task_queue)}")

    if not task_queue:
        print("❌ No tasks found. Exiting.")
        exit()

    # ----------------------------
    # Resume index
    # ----------------------------
    start_index = load_progress()
    print(f"🔁 Resuming from index {start_index}")

    broken_tasks = []

    # ----------------------------
    # Main execution loop
    # ----------------------------
    for idx in range(start_index, start_index + 40):
        task_type, task_id = task_queue[idx]
        print(f"\n▶️ [{idx}] Running {task_type}/{task_id}")

        steps_taken = 0
        success = 0

        # 🔁 Loop detection memory
        last_commands = []
        last_observations = []

        try:
            env, current_obs = initialize_alfworld_env(task_type, task_id)

        except KeyError as e:
            print(f"⚠️ Skipping broken task (PDDL error): {e}")
            broken_tasks.append((task_type, task_id, str(e)))
            save_progress(idx + 1)
            log_task_result(task_id, task_type, 0, 0)
            continue

        try:
            # ----------------------------
            # Extract task instruction
            # ----------------------------
            if "Your task is to:" in current_obs:
                task = "Your task is to:" + current_obs.split("Your task is to:")[-1].strip()
            else:
                task = "No task found."

            interaction_history = f"Observation:\n{current_obs.strip()}\n\n{task}"

            # ----------------------------
            # Agent loop
            # ----------------------------
            for step in range(1, MAX_STEPS + 1):
                print(f"\n{'='*25} STEP {step} {'='*25}")

                llm_output = get_next_action(interaction_history, few_shot_examples)

                # Extract reasoning
                try:
                    reasoning = re.search(
                        r"LLM Think:(.*?)(?=\nYour command:)",
                        llm_output,
                        re.DOTALL
                    ).group(1).strip()
                except AttributeError:
                    reasoning = "Recovering from malformed output."

                # Extract command
                try:
                    command = re.search(r"Your command:\s*(.*)", llm_output).group(1).strip()
                except AttributeError:
                    command = "look"

                print(f"LLM Think: {reasoning}")
                print(f"Your command: {command}")

#                 steps_taken += 1

                # ----------------------------
                # ConceptNet calls
                # ----------------------------
                tool_output = None

                if command.startswith("ConceptSearch"):
                    match = re.search(r"ConceptSearch\(['\"](.*?)['\"]\)", command)
                    if match:
                        tool_output = ConceptSearch(match.group(1))

                elif command.startswith("ConceptRelate"):
                    match = re.search(
                        r"ConceptRelate\(['\"](.*?)['\"],\s*['\"](.*?)['\"]\)",
                        command
                    )
                    if match:
                        tool_output = ConceptRelate(match.group(1), match.group(2))

                if tool_output:
                    interaction_history += f"\n\nLLM Think: {reasoning}\n{tool_output}"
                    print(tool_output)
                    continue

                # ----------------------------
                # Environment step
                # ----------------------------
                new_obs, done, infos = play_one_step(env, command)
                steps_taken += 1

                interaction_history += (
                    f"\n\nLLM Think: {reasoning}\n"
                    f"Your command: {command}\n\n"
                    f"Observation:\n{new_obs.strip()}"
                )

                print("\n" + "-" * 20 + " NEW OBSERVATION " + "-" * 20)
                print(new_obs.strip())
                print("-" * 57)

                # ----------------------------
                # Update loop memory
                # ----------------------------
                last_commands.append(command)
                last_observations.append(new_obs.strip())

                if len(last_commands) > 2:
                    last_commands.pop(0)
                if len(last_observations) > 2:
                    last_observations.pop(0)

                # ----------------------------
                # Loop detection
                # ----------------------------
                if (
                    len(last_commands) == 2 and
                    len(last_observations) == 2 and
                    last_commands[0] == last_commands[1] and
                    all(obs.lower() == "nothing happens." for obs in last_observations)
                ):
                    print("🔁 Non-ending loop detected. Forcing strategy change.")

                    interaction_history += (
                        "\n\n⚠️ LOOP DETECTED:\n"
                        "The last two identical actions resulted in 'Nothing happens.'.\n"
                        "You MUST choose a DIFFERENT valid action to escape this loop.\n"
                        "DO NOT repeat the same command again."
                    )

                # ----------------------------
                # Success detection
                # ----------------------------
                if "You won!" in new_obs:
                    print("🎉 Task completed!")
                    success = 1
                    break

                if done:
                    print("🏁 Environment ended episode.")
                    if any(k in new_obs for k in ["You move", "moved", "placed", "turn on", "clean", "cool"]):
                        success = 1
#                         print("new obs has researved keywords")
                    break

            # ----------------------------
            # Save log
            # ----------------------------
            os.makedirs(PREFIX, exist_ok=True)
            output_file = f"{PREFIX}/{idx + 1}-{Approach}--{task_type}---{task_id}.txt"
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(interaction_history)

            print(f"✅ Log saved: {output_file}")


        finally:
            try:
                env.close()
            except:
                pass

            log_task_result(task_id, task_type, success, steps_taken)
            save_progress(idx + 1)


    # ----------------------------
    # Summary
    # ----------------------------
    print("\n🏁 All tasks processed.")
    print(f"❌ Broken tasks skipped: {len(broken_tasks)}")



In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt

def print_and_save_stats_as_image(csv_file, output_image=f"{Approach}_{PREFIX}.png"):
    if not os.path.exists(csv_file):
        print(f"❌ CSV file '{csv_file}' not found.")
        return

    df = pd.read_csv(csv_file)

    if df.empty:
        print("⚠️ CSV file is empty.")
        return

    # ------------------------------------------------
    # Exclude broken tasks (success == 0 AND steps == 0)
    # ------------------------------------------------
    filtered_df = df[~((df['success'] == 0) & (df['steps'] == 0))]

    if filtered_df.empty:
        print("⚠️ No valid (non-broken) tasks found.")
        return

    total_tasks = len(filtered_df)
    completed_tasks = int(filtered_df['success'].sum())
    failed_tasks = total_tasks - completed_tasks
    success_ratio = (completed_tasks / total_tasks) * 100

    avg_steps_overall = filtered_df['steps'].mean()
    avg_steps_success = filtered_df[filtered_df['success'] == 1]['steps'].mean()
    avg_steps_failed = filtered_df[filtered_df['success'] == 0]['steps'].mean()

    # ------------------------------------------------
    # Create summary table
    # ------------------------------------------------
    summary_df = pd.DataFrame({
        "Metric": [
            "Total valid tasks",
            "Completed tasks",
            "Failed tasks",
            "Success ratio (%)",
            "Avg steps per task"
        ],
        "Value": [
            total_tasks,
            completed_tasks,
            failed_tasks,
            f"{success_ratio:.2f}",
            f"{avg_steps_overall:.2f}"

        ]
    })

    # ------------------------------------------------
    # Print to console
    # ------------------------------------------------
    print("\n📊 Task Performance Summary (Broken Tasks Excluded)")
    print("-" * 50)
    print(summary_df.to_string(index=False))
    print("-" * 50)

    # ------------------------------------------------
    # Save as image
    # ------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axis('off')

    table = ax.table(
        cellText=summary_df.values,
        colLabels=summary_df.columns,
        cellLoc='center',
        loc='center'
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)

    # ---- Titles (CORRECT WAY) ----
    plt.suptitle(
        "Task Performance Summary (Broken Tasks Excluded)",
        fontsize=14,
        fontweight="bold",
        y=0.97
    )

    ax.text(
        0.5, 0.90,
        f"Task: {PREFIX}",
        ha="center",
        va="center",
        fontsize=10,
        transform=ax.transAxes
    )

    ax.text(
        0.5, 0.85,
        f"Approach:{Approach}",
        ha="center",
        va="center",
        fontsize=10,
        transform=ax.transAxes
    )

    plt.tight_layout(rect=[0, 0, 1, 0.88])
    plt.savefig(output_image, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"🖼️ Summary image saved to: {os.path.abspath(output_image)}\n")

print_and_save_stats_as_image(CSV_LOG_FILE)